# Percobaan KNN Berbobot Fuzzy

Notebook ini memakai **data dummy** untuk menguji `src/knn_fuzzy.py` sebelum data asli dari Anggota 1 dan hasil FCM dari Anggota 2 tersedia.

Tujuannya:
- Menghitung rekomendasi dengan **α = 1** (KNN biasa, hanya fitur) dan **α = 0,5** (fitur + keanggotaan).
- Membuktikan bahwa hasil α = 1 sama dengan `NearestNeighbors` dari scikit-learn.


## 1. Siapkan data dummy

Dibuat 20 film dengan 4 fitur acak di rentang 0–1, plus matriks keanggotaan 3 cluster. Setiap baris `U` dinormalisasi agar jumlahnya 1, seperti keluaran FCM yang sah. Seed 42 dipakai supaya hasilnya bisa diulang.


In [1]:
import sys
from pathlib import Path

# Notebook bisa dijalankan dari folder notebooks/ atau dari akar proyek.
for kandidat in [Path.cwd(), Path.cwd().parent]:
    if (kandidat / "src" / "knn_fuzzy.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import numpy as np
from sklearn.neighbors import NearestNeighbors

from src.knn_fuzzy import combined_distance, recommend, membership_new, recommend_new

rng = np.random.default_rng(42)
X = rng.random((20, 4))          # 20 film, 4 fitur (sudah dalam rentang 0–1)
U = rng.random((20, 3))
U = U / U.sum(axis=1, keepdims=True)  # keanggotaan 3 cluster, tiap baris jumlahnya 1

print("Bentuk X:", X.shape)
print("Bentuk U:", U.shape)
print("Jumlah per baris U (harus 1):", np.round(U.sum(axis=1), 6)[:5], "...")
print("Contoh U[0]:", np.round(U[0], 3))


Bentuk X: (20, 4)
Bentuk U: (20, 3)
Jumlah per baris U (harus 1): [1. 1. 1. 1. 1.] ...
Contoh U[0]: [0.352 0.216 0.432]


## 2. Bandingkan rekomendasi α = 1 dan α = 0,5

Film ke-0 dijadikan kueri, lalu diminta 5 tetangga terdekat. Jika urutan hasilnya berbeda, artinya informasi keanggotaan FCM memang mengubah rekomendasi.


In [2]:
query_idx = 0
k = 5

idx_1, dist_1 = recommend(X, U, query_idx, k=k, alpha=1.0)
idx_05, dist_05 = recommend(X, U, query_idx, k=k, alpha=0.5)

print("α = 1.0  (hanya fitur)")
print("  indeks :", idx_1)
print("  jarak  :", np.round(dist_1, 4))
print()
print("α = 0.5  (fitur + keanggotaan)")
print("  indeks :", idx_05)
print("  jarak  :", np.round(dist_05, 4))
print()
print("Film kueri ada di hasil? ", query_idx in idx_1 or query_idx in idx_05)
print("Urutan α = 1 dan α = 0,5 sama?", np.array_equal(idx_1, idx_05))


α = 1.0  (hanya fitur)
  indeks : [ 5 13 15  4 16]
  jarak  : [0.2414 0.2859 0.3206 0.4405 0.5263]

α = 0.5  (fitur + keanggotaan)
  indeks : [13  4 15  5 10]
  jarak  : [0.2076 0.325  0.3338 0.3391 0.4154]

Film kueri ada di hasil?  False
Urutan α = 1 dan α = 0,5 sama? False


## 3. Periksa α = 1 terhadap scikit-learn

Jika fungsi jarak fitur benar, 5 tetangga pada α = 1 harus identik dengan keluaran `NearestNeighbors`. Tetangga pertama dari scikit-learn adalah film kueri itu sendiri, jadi dilewati.


In [3]:
model = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
model.fit(X)
jarak_sk, indeks_sk = model.kneighbors(X[[query_idx]])

print("sklearn (termasuk diri sendiri):", indeks_sk[0], np.round(jarak_sk[0], 4))
print("sklearn tanpa diri sendiri    :", indeks_sk[0, 1:], np.round(jarak_sk[0, 1:], 4))
print("recommend() dengan α = 1      :", idx_1, np.round(dist_1, 4))

cocok_indeks = np.array_equal(idx_1, indeks_sk[0, 1:])
cocok_jarak = np.allclose(dist_1, jarak_sk[0, 1:])
print()
print("Indeks cocok?", cocok_indeks)
print("Jarak  cocok?", cocok_jarak)


sklearn (termasuk diri sendiri): [ 0  5 13 15  4 16] [0.     0.2414 0.2859 0.3206 0.4405 0.5263]
sklearn tanpa diri sendiri    : [ 5 13 15  4 16] [0.2414 0.2859 0.3206 0.4405 0.5263]
recommend() dengan α = 1      : [ 5 13 15  4 16] [0.2414 0.2859 0.3206 0.4405 0.5263]

Indeks cocok? True
Jarak  cocok? True


## 4. Keanggotaan untuk film baru

Film yang tidak ikut proses FCM belum punya vektor `u`. Nilainya dihitung dari jarak film itu ke pusat-pusat cluster `V` memakai rumus pembaruan keanggotaan FCM. Di sini `V` dibuat acak hanya untuk peragaan; nanti diganti pusat cluster asli dari Anggota 2.


In [4]:
# 3 pusat cluster acak di ruang 4 fitur (sama seperti X)
V = rng.random((3, 4))
x_baru = rng.random(4)

u_baru = membership_new(x_baru, V, m=2.0)
print("x_baru :", np.round(x_baru, 3))
print("u_baru :", np.round(u_baru, 3), "  jumlah =", round(u_baru.sum(), 6))

idx_baru, dist_baru = recommend_new(x_baru, X, U, V, k=5, alpha=0.5, m=2.0)
print()
print("Rekomendasi untuk film baru (α = 0,5):")
print("  indeks :", idx_baru)
print("  jarak  :", np.round(dist_baru, 4))


x_baru : [0.14  0.554 0.109 0.672]
u_baru : [0.201 0.622 0.177]   jumlah = 1.0

Rekomendasi untuk film baru (α = 0,5):
  indeks : [19 17  2 14 16]
  jarak  : [0.3486 0.3776 0.4099 0.4127 0.4335]
